# SpectralGuard Phase 3 — Automatic H100 Experiments
Prepare and verify the cumulative handoff on CPU first. Only after the terminal prints `PHASE 3 RESUME VERIFICATION: READY FOR H100`, select an H100 runtime and click **Run all** once. The notebook verifies the pinned handoff again, resumes E01 at exactly 3,507/6,000 rows, uses exact-length Mamba batches, masked Transformer batches, automatic OOM backoff, atomic checkpoints, and a three-hour paid-session timebox with export reserve. Completed evidence is never repeated. A complete run creates `Experiments/exports/phase3_experiments.zip`; otherwise it creates a new cumulative handoff.

In [ ]:
from pathlib import Path
import os
import importlib.util
import subprocess
import sys
if not Path('requirements-lightning.txt').is_file():
    os.chdir('..')
assert Path('requirements-lightning.txt').is_file(), f'Repository root not found from {Path.cwd()}'
handoff_sha256 = 'c2b113786016e23b61f3863373260569cf45a1d3a82769d2291b13376f7ad7fb'
subprocess.run([sys.executable, 'scripts/verify_phase3_resume.py', '--expected-sha256', handoff_sha256, '--expected-e01-rows', '3507'], check=True)
assert sys.version_info[:2] == (3, 12), f'Lightning Phase 3 requires Python 3.12, found {sys.version}'
if importlib.util.find_spec('pip') is None:
    subprocess.run([sys.executable, '-m', 'ensurepip', '--upgrade'], check=True)
%pip install -q -r requirements-lightning.txt
%pip install -q -e . --no-deps
subprocess.run([sys.executable, 'scripts/install_lightning_mamba_kernels.py', '--cuda-smoke'], check=True)
print('Finished')

In [ ]:
import importlib.metadata as md
import torch
torch_version = md.version('torch').split('+')[0]
assert torch_version == '2.8.0', f'Expected torch 2.8.0, found {md.version("torch")}'
assert str(torch.version.cuda).startswith('12.'), f'Expected a CUDA 12.x torch build, found {torch.version.cuda}'
assert torch.cuda.is_available(), 'STOP: select an H100 runtime before Run all.'
gpu = torch.cuda.get_device_name(0)
assert 'H100' in gpu.upper(), f'STOP: H100 required, detected {gpu}'
assert torch.cuda.is_bf16_supported(), 'STOP: BF16 support is required.'
print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': gpu, 'vram_gib': torch.cuda.get_device_properties(0).total_memory / 1024**3})
print('Finished')

In [ ]:
from src.experiments.phase3_runner import run_automatic_phase3
automatic_report = run_automatic_phase3()
assert automatic_report['status'] in {'COMPLETE', 'TIMEBOX_HANDOFF'}
print({key: automatic_report.get(key) for key in ('status', 'session_elapsed_seconds', 'experiments_zip')})
print('Finished')

In [ ]:
from pathlib import Path
complete = automatic_report['status'] == 'COMPLETE'
result = Path('Experiments/exports/phase3_experiments.zip' if complete else 'Experiments/exports/phase3_handoff.tar.gz').resolve()
assert result.is_file() and result.stat().st_size > 0
print(f'DOWNLOAD READY: {result}')
print('Next: stop the H100 and run Notebook 2 on CPU.' if complete else 'Timebox reached safely: stop the H100 and upload this handoff to the next account.')
print('Finished')